# Numerical feature selection

Scores numeric feature candidates against the listing's first asking rent using normalized Gini, grouped gradient boosting with permutation importance, and Lasso. A synthetic answer key provides a separate recovery check.

**Project step:** §4.2 step 1.0.

**Inputs:** `outputs/02_phase_01_feature_funnel{suffix}.csv`, master table and column catalog.

**Outputs:** `outputs/03_phase_01_feature_funnel_numerical{suffix}.csv`, `outputs/03_phase_01_feature_selection_numerical{suffix}.csv`.

**Run after:** 02_feature_funnel.

**Runtime:** ~10 min.


In [1]:
# ---- Data ----
INPUT = "data/master_listing.parquet"   # relative to the repo root
OUTPUT_DIR = "outputs"                  # every CSV / XLSX / PNG goes here

# ---- Filters (None = no filter) ----
CITY = None             # "chicago", "dfw", "dc", or a list of them       -> column `market`
NEIGHBORHOOD = None     # submarket/list -> column `market_building__submarket`
START_DATE = None       # "YYYY-MM-DD": keep listings with DATE_COLUMN >= START_DATE
END_DATE = None         # "YYYY-MM-DD": keep listings with DATE_COLUMN <= END_DATE
DATE_COLUMN = "market_unit__enter_market"

# ---- Selection ----
TARGET = "market_price_observation__first_asking_rent"
ANSWER_KEY = "truth_floorplan_market_rent__true_market_rent_at_enter"
EXCLUDE_PREFIXES = ["pms_", "market_price_observation__", "market_concession_event__", "truth_"]
EXCLUDE_COLUMNS = ["market_unit__days_on_market"]
SAMPLE = 300_000
N_FOLDS = 3
SEED = 42
GINI_MIN, GB_MIN = 0.10, 0.001


In [2]:
from pathlib import Path

import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / INPUT
CATALOG = ROOT / "data" / "master_columns.csv"
OUTPUTS = ROOT / OUTPUT_DIR
OUTPUTS.mkdir(parents=True, exist_ok=True)
NN = 3
FILTER_COLUMNS = ["market", "market_building__submarket", DATE_COLUMN]


def _as_list(value):
    """Normalize one filter value or a list of values."""
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)


def suffix():
    """Build the shared output suffix from the active filters."""
    parts = []
    if CITY is not None:
        parts.append("+".join(map(str, _as_list(CITY))))
    if NEIGHBORHOOD is not None:
        parts.append("+".join(map(str, _as_list(NEIGHBORHOOD))))
    if START_DATE is not None or END_DATE is not None:
        parts.append(f"{START_DATE or 'start'}_to_{END_DATE or 'end'}")
    return "" if not parts else "__" + "__".join(parts)


def out(name, ext="csv"):
    """Build an output path with this notebook's number and active filters."""
    return OUTPUTS / f"{NN:02d}_phase_01_{name}{suffix()}.{ext}"


def upstream(number, name, ext="csv"):
    """Build an upstream notebook's output path with the active filters."""
    return OUTPUTS / f"{number:02d}_phase_01_{name}{suffix()}.{ext}"


def need(path, notebook):
    """Raise a clear instruction when a required upstream output is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Run {notebook} first; missing {path}")


def load(columns):
    """Read requested master columns and apply the configured listing filters."""
    requested = list(dict.fromkeys(columns))
    read_columns = list(dict.fromkeys(requested + FILTER_COLUMNS))
    frame = pd.read_parquet(MASTER, columns=read_columns)
    valid_values = {
        column: sorted(frame[column].dropna().astype(str).unique())
        for column in ("market", "market_building__submarket")
    }
    for column, value, label in (
        ("market", CITY, "city"),
        ("market_building__submarket", NEIGHBORHOOD, "neighborhood"),
    ):
        if value is not None:
            wanted = list(map(str, _as_list(value)))
            valid = valid_values[column]
            unknown = sorted(set(wanted) - set(valid))
            if unknown:
                raise ValueError(f"Unknown {label}(s) {unknown}. Valid values: {valid}")
            before = len(frame)
            frame = frame[frame[column].astype(str).isin(wanted)]
            note(f"{label}: {before:,} → {len(frame):,} rows")
    if START_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] >= pd.Timestamp(START_DATE)]
        note(f"start date: {before:,} → {len(frame):,} rows")
    if END_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] <= pd.Timestamp(END_DATE)]
        note(f"end date: {before:,} → {len(frame):,} rows")
    return frame[requested]


## Feature selection

Scores eligible numerical candidates against asking rent, the Phase 1 v1 target available when a unit is listed, using three methods. The answer key is used only to check whether feature rankings recover known synthetic market rent. The selection CSV is sorted by gradient boosting importance:

| column | method | how to read it |
| --- | --- | --- |
| `gini` | Credit-scoring Gini for a continuous target (normalized Gini of the Lorenz curve), on every listing where the feature is present | 1 = ranks rent perfectly, 0 = no better than random, negative = higher value means lower rent. Rank by absolute value. For a 0/1 target it equals 2·AUC − 1. |
| `gradient_boosting` | R² lost on unseen buildings when the feature is shuffled (permutation importance of a gradient-boosted model) | about 0 = the model does not need it. Counts only what the feature adds beyond the others. |
| `gini_answer_key` | Gini of each feature against `ANSWER_KEY`, where both values are present | Checks rankings against synthetic ground truth; never an input target. |
| `lasso` | Coefficient of an L1-penalized linear regression on standardized features | change in rent ($) per one standard deviation of the feature. Exactly 0 = Lasso dropped it. |

Gini looks at one feature at a time; gradient boosting and Lasso look at all of them together,
so a feature that repeats another one can have a high Gini and a low score on the other two.

- A low-completeness feature's Gini only covers the rows it is present on: read it next to `completeness_pct`.
- Both models are validated with `GroupKFold` by `property_id`, so they are scored on buildings they never saw.
- Candidates exclude answer keys, labels, leaky values, PMS columns and post-listing price or concession observations. Building history averages are called out later because they may include future information in a backtest.
- `GINI_MIN` and `GB_MIN` only drive the printed summary; the CSV keeps the raw scores.

In [3]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

def gini(x, y):
    """Normalized Gini of ordering y by x.

    Sort by x from high to low, take the Gini of the Lorenz curve of cumulative y, and divide by
    the Gini of the perfect ordering (y by itself). Tied x values are taken as one block, so a
    feature with few distinct values (bed, bath) is not scored on an arbitrary tie order.
    """
    def lorenz_gini(key):
        g = pd.DataFrame({"k": key, "y": y}).groupby("k")["y"].agg(["size", "sum"]).iloc[::-1]
        px = np.r_[0, g["size"].cumsum() / len(y)]
        py = np.r_[0, g["sum"].cumsum() / y.sum()]
        return 2 * ((py[1:] + py[:-1]) / 2 * np.diff(px)).sum() - 1
    return lorenz_gini(x) / lorenz_gini(y)


FUNNEL_PATH = upstream(2, "feature_funnel")
need(FUNNEL_PATH, "02_feature_funnel")
step(f"Reading {FUNNEL_PATH.relative_to(ROOT)}")
funnel = pd.read_csv(FUNNEL_PATH)
catalog = pd.read_csv(CATALOG)
num = funnel[funnel["type"].eq("number")].merge(
    catalog[["column", "role"]], left_on="variable", right_on="column", how="left"
).drop(columns="column")
GROUP_KEYS = ["unit_id", "property_id"]
step(f"Loading numerical columns from {MASTER.name} on filtered listings")
df_num = load(GROUP_KEYS + [TARGET, ANSWER_KEY] + num["variable"].tolist())
done(f"{df_num.shape[0]:,} filtered rows x {df_num.shape[1]} columns")

num["x_status"] = "candidate"
rule_columns = ["role", "prefix", "column", "empty", "constant"]
dropped = {rule: [] for rule in rule_columns}
for row_index, row in num.iterrows():
    variable = row["variable"]
    reason = None
    if row["role"] not in ("feature", "inferred"):
        reason = f"excluded: role {row['role']}"
        rule = "role"
    else:
        prefix = next((item for item in EXCLUDE_PREFIXES if variable.startswith(item)), None)
        if prefix is not None:
            reason = f"excluded: {prefix} prefix"
            rule = "prefix"
        elif variable in EXCLUDE_COLUMNS:
            reason = "excluded: explicit column"
            rule = "column"
        elif row["completeness_pct"] <= 0:
            reason = "excluded: empty"
            rule = "empty"
        elif df_num[variable].nunique(dropna=True) <= 1:
            reason = "excluded: constant"
            rule = "constant"
    if reason is not None:
        num.loc[row_index, "x_status"] = reason
        dropped[rule].append(variable)
for rule, names in dropped.items():
    note(f"dropped by {rule}: {len(names)}" + (f" — {', '.join(names)}" if names else ""))
X_cols = num.loc[num["x_status"].eq("candidate"), "variable"].tolist()
num.to_csv(out("feature_funnel_numerical"), index=False)
done(f"{len(X_cols)} eligible candidates from {len(num)} numerical variables")
scored = X_cols
data = df_num[df_num[TARGET].notna()].copy()
done(f"{len(data):,} filtered listings with a known asking rent")

step("Gini of each candidate vs asking rent and answer key")
scores = pd.DataFrame(index=scored)
gini_scores = []
answer_scores = []
for feature in scored:
    present = data[feature].notna() & data[TARGET].notna()
    gini_scores.append(gini(data.loc[present, feature].to_numpy(), data.loc[present, TARGET].to_numpy()))
    answer_present = df_num[feature].notna() & df_num[ANSWER_KEY].notna()
    answer_scores.append(gini(df_num.loc[answer_present, feature].to_numpy(),
                              df_num.loc[answer_present, ANSWER_KEY].to_numpy()))
scores["gini"] = gini_scores
scores["gini_answer_key"] = answer_scores
done(f"|Gini| >= {GINI_MIN} for {(scores['gini'].abs() >= GINI_MIN).sum()} of {len(scored)} candidates")



▶ Reading outputs/02_phase_01_feature_funnel.csv

▶ Loading numerical columns from master_listing.parquet on filtered listings


  ✓ 967,661 filtered rows x 110 columns


  • dropped by role: 34 — truth_lease_event__true_term_months, truth_lease_event__true_rent, truth_lease_event__true_concession_monthly, truth_lease_event__true_effective_rent, truth_lease_event__true_market_rent_at_signing, truth_building_params__quality_true, truth_building_params__amenity_score, truth_building_params__price_bias, truth_building_params__elasticity, truth_building_params__eps_building, truth_building_params__number_units_true, truth_building_params__year_built_true, truth_building_params__number_stories_true, truth_building_params__lead_base_rate, market_price_observation__first_asking_rent, market_price_observation__last_asking_rent, market_price_observation__min_asking_rent, market_price_observation__max_asking_rent, market_price_observation__day_weighted_asking_rent, market_price_observation__first_effective_rent, market_price_observation__last_effective_rent, market_price_observation__day_weighted_effective_rent, market_price_observation__first_asking_rent_psf, ma

  ✓ 967,277 filtered listings with a known asking rent

▶ Gini of each candidate vs asking rent and answer key


  ✓ |Gini| >= 0.1 for 34 of 44 candidates


In [4]:
step(f"Gradient boosting: {N_FOLDS}-fold GroupKFold by building on {SAMPLE:,} sampled listings (~3 min)")
# Grouped by property_id so no building is in both train and validation: otherwise the
# building columns would score well just by letting the model memorize buildings.
sample = data.sample(min(SAMPLE, len(data)), random_state=SEED)
X = sample[scored].astype("float64")
y = sample[TARGET].to_numpy()
folds = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups=sample["property_id"]))

importances = []
for i, (train, valid) in enumerate(folds, 1):
    model = HistGradientBoostingRegressor(max_iter=300, random_state=SEED).fit(X.iloc[train], y[train])
    score = model.score(X.iloc[valid], y[valid])
    note(f"fold {i}/{N_FOLDS}: R² on unseen buildings = {score:.3f}, "
         "shuffling features ...")
    perm = permutation_importance(model, X.iloc[valid], y[valid], scoring="r2",
                                  n_repeats=3, random_state=SEED, n_jobs=-1)
    importances.append(perm.importances_mean)
scores["gradient_boosting"] = np.mean(importances, axis=0)
done(f"importance > {GB_MIN} for {(scores['gradient_boosting'] > GB_MIN).sum()} of {len(scored)} features")


▶ Gradient boosting: 3-fold GroupKFold by building on 300,000 sampled listings (~3 min)


  • fold 1/3: R² on unseen buildings = 0.801, shuffling features ...


  • fold 2/3: R² on unseen buildings = 0.796, shuffling features ...


  • fold 3/3: R² on unseen buildings = 0.777, shuffling features ...


  ✓ importance > 0.001 for 24 of 44 features


In [5]:
step("Lasso: L1-penalized linear regression on standardized features")
# Median-imputed and standardized, so each coefficient is the change in rent ($) for one
# standard deviation of the feature. Alpha is the largest one whose CV error is within one
# standard error of the best (the "1-SE rule"): the sparsest model that is about as good.
Xs = StandardScaler().fit_transform(X.fillna(X.median()))
cv = LassoCV(alphas=50, cv=folds, max_iter=5000, n_jobs=-1).fit(Xs, y)
mse = cv.mse_path_.mean(axis=1)
se = cv.mse_path_.std(axis=1) / np.sqrt(N_FOLDS)
alpha = cv.alphas_[mse <= mse.min() + se[mse.argmin()]].max()
scores["lasso"] = Lasso(alpha=alpha, max_iter=5000).fit(Xs, y).coef_
kept_count = (scores["lasso"] != 0).sum()
done(f"alpha = {alpha:.2f}: Lasso kept {kept_count} of {len(scored)} features; "
     "the rest are exactly 0")


▶ Lasso: L1-penalized linear regression on standardized features


  ✓ alpha = 13.10: Lasso kept 23 of 44 features; the rest are exactly 0


In [6]:
step(f"Writing {out('feature_selection_numerical').relative_to(ROOT)}")
selection = (num.loc[num["x_status"].eq("candidate"),
                      ["source_table", "variable", "completeness_pct"]]
             .rename(columns={"variable": "feature"})
             .set_index("feature")
             # + 0.0 turns Lasso's -0.0 into 0.0
             .join(scores.round({"gini": 4, "gradient_boosting": 4, "lasso": 2,
                                 "gini_answer_key": 4}) + 0.0)
             .sort_values("gradient_boosting", ascending=False, na_position="last")
             .reset_index())
selection = selection[["source_table", "feature", "completeness_pct", "gini",
                     "gradient_boosting", "lasso", "gini_answer_key"]]
selection.to_csv(out("feature_selection_numerical"), index=False)
done(f"{len(selection)} candidates x {selection.shape[1]} score columns")

kept = {
    f"|gini| >= {GINI_MIN}": selection["gini"].abs() >= GINI_MIN,
    f"gradient boosting > {GB_MIN}": selection["gradient_boosting"] > GB_MIN,
    "lasso != 0": selection["lasso"].fillna(0) != 0,
}
all_three = np.logical_and.reduce(list(kept.values()))

step("Funnel")
for label, count in [("candidate features", len(selection)), ("non-empty", len(scored))] + [
    (label, int(mask.sum())) for label, mask in kept.items()
] + [("kept by all three", int(all_three.sum()))]:
    proportion = count / max(len(selection), 1)
    color = GREEN if proportion >= 0.5 else YELLOW if proportion >= 0.25 else RED
    print(f"  {label:28s}{color}{'█' * round(40 * count / max(len(selection), 1)):40s} {count:>3}{RESET}")

step("Kept by all three methods")
for row in selection[all_three].itertuples():
    print(f"{GREEN}  ✓ {row.feature:52s}{RESET} gini {row.gini:+.3f}   "
          f"gb {row.gradient_boosting:.4f}   lasso {row.lasso:+9.2f}   "
          f"answer-key Gini {row.gini_answer_key:+.3f}")
selection.head(25)


▶ Writing outputs/03_phase_01_feature_selection_numerical.csv
  ✓ 44 candidates x 7 score columns

▶ Funnel
  candidate features          ████████████████████████████████████████  44
  non-empty                   ████████████████████████████████████████  44
  |gini| >= 0.1               ███████████████████████████████           34
  gradient boosting > 0.001   ██████████████████████                    24
  lasso != 0                  █████████████████████                     23
  kept by all three           ███████████████                           17

▶ Kept by all three methods
  ✓ market_unit__sqft                                    gini +0.538   gb 0.3596   lasso   +306.23   answer-key Gini +0.542
  ✓ market_building__n_building_amenities                gini +0.509   gb 0.1958   lasso   +156.45   answer-key Gini +0.505
  ✓ market_building__lon                                 gini +0.395   gb 0.1411   lasso   +278.78   answer-key Gini +0.402
  ✓ market_building__year_built         

,source_table,feature,completeness_pct,gini,gradient_boosting,lasso,gini_answer_key
0,market_unit,market_unit__sqft,95.67,0.5376,0.3596,306.23,0.5425
1,market_building,market_building__lat,100.00,0.1906,0.2814,0.00,0.1883
2,market_building,market_building__n_building_amenities,82.79,0.5090,0.1958,156.45,0.5055
3,market_building,market_building__lon,100.00,0.3951,0.1411,278.78,0.4019
4,market_building,market_building__year_built,89.22,0.1063,0.0534,73.08,0.1129
5,market_unit,market_unit__bed,100.00,0.3713,0.0415,0.00,0.3758
6,market_building,market_building__avg_price_change,100.00,-0.1244,0.0153,-19.55,-0.1261
7,market_building,market_building__avg_time_on_market_days,99.99,0.0778,0.0121,38.95,0.0316
8,market_leasing_activity_daily,market_leasing_activity_daily__leased_pct_at_e...,90.49,0.0146,0.0106,0.00,0.0217
9,market_building,market_building__n_unit_amenities,82.79,0.5015,0.0101,36.86,0.4964
